In [1]:
import torch
import torch.nn as nn

# 1. Initialize device and standard architecture module
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Sequential(
    nn.Linear(100, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

# 2. Instantiate the dynamic Gradient Scaler
# If using BF16, a GradScaler is unnecessary because BF16 does not suffer from underflow!
scaler = torch.amp.GradScaler(device="cuda", enabled=(device.type == "cuda"))

# Generate mock training vectors
mock_X = torch.randn(64, 100).to(device)
mock_y = torch.randint(0, 10, (64,)).to(device)
criterion = nn.CrossEntropyLoss()

model.train()
optimizer.zero_grad()

# 3. Execute the forward pass within the Autocast context manager
# Autocast automatically switches operations to float16/bfloat16 when performance gains exist
with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
    outputs = model(mock_X)
    loss = criterion(outputs, mock_y)

# 4. Scale the loss and compute gradients via backpropagation
# scaler.scale(loss) multiplies the loss to prevent underflow errors
scaler.scale(loss).backward()

# 5. Unscale gradients and update the FP32 master parameters safely
# scaler.step() internally checks for NaNs/Infs before calling optimizer.step()
scaler.step(optimizer)

# 6. Update the dynamic scaling factor for the next training iteration
scaler.update()

print("Mixed precision training iteration executed safely.")
print(f"Current scale factor status: {scaler.get_scale()}")

Mixed precision training iteration executed safely.
Current scale factor status: 1.0


/tmp/ipykernel_917/3205287281.py:28: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
